# Chương 1: Thực hành - So sánh AI Ký hiệu (Symbolic AI) vs Học máy Thống kê (Statistical Learning)

**Mục tiêu bài thực hành:**
1. Xây dựng một **Hệ chuyên gia dựa trên luật (Symbolic Rule-Based Expert System)** kinh điển với các mệnh đề logic IF-THEN.
2. Tự cài đặt từ đầu (**from scratch**) một **Mô hình Học máy Thống kê (Statistical Logistic Classifier)** sử dụng thuật toán Giảm dần đạo hàm (Gradient Descent) chỉ với `numpy`.
3. Xây dựng một **Hệ thống Lai (Neuro-Symbolic Hybrid System)** kết hợp suy diễn xác suất với các luật an toàn cứng (Safety Guardrails).
4. Phân tích thực nghiệm: Trực quan hóa ranh giới quyết định (Decision Boundary), kiểm tra độ giòn (Brittleness) trước nhiễu và dữ liệu lệch phân phối (Out-Of-Distribution).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Thiết lập hạt giống ngẫu nhiên để tái lập kết quả
np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

## 1. Bài toán và Tạo dữ liệu mô phỏng (Synthetic Dataset)

Giả sử ta cần thẩm định hồ sơ xin vay vốn của khách hàng dựa trên hai đặc trưng chuẩn hóa:
- $X_1$: Thu nhập hàng tháng (thang điểm $0 - 10$).
- $X_2$: Điểm lịch sử tín dụng (thang điểm $0 - 10$).
- Nhãn mục tiêu $Y \in \{0, 1\}$:
  - $Y = 1$: Rủi ro vỡ nợ (Từ chối cho vay).
  - $Y = 0$: Khả năng thanh toán tốt (Phê duyệt cho vay).

In [ ]:
def generate_credit_data(n_samples=300, noise_ratio=0.08):
    """
    Sinh dữ liệu hồ sơ tín dụng với một ranh giới thực tế tiềm ẩn kèm nhiễu ngẫu nhiên.
    """
    # Sinh ngẫu nhiên điểm thu nhập và điểm tín dụng
    X = np.random.uniform(0.5, 9.5, size=(n_samples, 2))
    
    # Ranh giới lý tưởng: nếu 1.2*X1 + 1.5*X2 < 12 => Rủi ro vỡ nợ (1), ngược lại An toàn (0)
    latent_score = 1.2 * X[:, 0] + 1.5 * X[:, 1]
    y = (latent_score < 12.0).astype(int)
    
    # Thêm nhiễu ngẫu nhiên mô phỏng sự bất định đời thực
    n_noise = int(n_samples * noise_ratio)
    flip_indices = np.random.choice(n_samples, size=n_noise, replace=False)
    y[flip_indices] = 1 - y[flip_indices]
    
    return X, y

X_train, y_train = generate_credit_data(n_samples=300, noise_ratio=0.08)
X_test, y_test = generate_credit_data(n_samples=100, noise_ratio=0.08)

print(f"Kích thước tập huấn luyện: X = {X_train.shape}, y = {y_train.shape}")
print(f"Tỷ lệ nhãn vỡ nợ (Y=1): {np.mean(y_train)*100:.1f}%")

## 2. Hệ chuyên gia Ký hiệu (Symbolic Rule-Based AI)

Hệ thống ký hiệu sử dụng các quy tắc logic tường minh do con người xây dựng:
- **Luật 1 (Rủi ro cao):** Nếu $X_1 < 3.5$ và $X_2 < 4.0 \implies$ Dự đoán $Y = 1$.
- **Luật 2 (An toàn cao):** Nếu $X_1 \ge 6.0$ hoặc $X_2 \ge 7.0 \implies$ Dự đoán $Y = 0$.
- **Luật 3 (Bổ sung):** Nếu $X_1 + X_2 < 9.0 \implies$ Dự đoán $Y = 1$, ngược lại $Y = 0$.
- Nếu dữ liệu không thỏa các điều kiện luật chuyên gia, hệ thống rơi vào trạng thái bế tắc hoặc phải dựa vào luật mặc định.

In [ ]:
class SymbolicRuleExpertSystem:
    """
    Hệ thống chuyên gia ký hiệu dựa trên các quy tắc IF-THEN tường minh.
    """
    def __init__(self):
        self.name = "Symbolic Rule-Based Engine"
        
    def predict_single(self, x):
        x1, x2 = x[0], x[1]
        
        # Tập luật tiên nghiệm
        if x1 < 3.5 and x2 < 4.0:
            return 1, "Rule 1: Thu nhập thấp và lịch sử tín dụng kém -> Từ chối"
        elif x1 >= 6.0 or x2 >= 7.0:
            return 0, "Rule 2: Thu nhập cao hoặc tín dụng rất tốt -> Phê duyệt"
        elif (x1 + x2) < 9.0:
            return 1, "Rule 3: Tổng điểm tín dụng và thu nhập dưới ngưỡng an toàn -> Từ chối"
        else:
            return 0, "Rule 4 (Default): Đạt ngưỡng tối thiểu -> Phê duyệt"
            
    def predict(self, X):
        preds = [self.predict_single(x)[0] for x in X]
        return np.array(preds)

symbolic_model = SymbolicRuleExpertSystem()
sym_preds_test = symbolic_model.predict(X_test)
sym_acc = np.mean(sym_preds_test == y_test)
print(f"[Symbolic AI] Độ chính xác trên tập kiểm thử: {sym_acc * 100:.2f}%")

## 3. Trí tuệ nhân tạo Thống kê (Statistical Machine Learning)

Mô hình học máy thống kê học hàm dự đoán xác suất $\hat{y} = P(Y=1 \mid \mathbf{x})$ thông qua việc tối thiểu hóa hàm mất mát nhị phân (Binary Cross-Entropy Loss):
$$\hat{y} = \sigma(\mathbf{w}^T \mathbf{x} + b) = \frac{1}{1 + e^{-(\mathbf{w}^T \mathbf{x} + b)}}$$
$$\mathcal{L}(\mathbf{w}, b) = -\frac{1}{N} \sum_{i=1}^N \left[ y_i \log(\hat{y}_i) + (1 - y_i) \log(1 - \hat{y}_i) \right]$$

Gradient đạo hàm riêng để cập nhật tham số:
$$\frac{\partial \mathcal{L}}{\partial \mathbf{w}} = \frac{1}{N} \mathbf{X}^T (\hat{\mathbf{y}} - \mathbf{y}), \quad \frac{\partial \mathcal{L}}{\partial b} = \frac{1}{N} \sum_{i=1}^N (\hat{y}_i - y_i)$$

In [ ]:
class StatisticalLogisticClassifier:
    """
    Mô hình phân loại thống kê (Logistic Regression) tự cài đặt từ đầu bằng NumPy.
    """
    def __init__(self, lr=0.05, n_epochs=1000):
        self.lr = lr
        self.n_epochs = n_epochs
        self.w = None
        self.b = None
        self.losses = []
        
    @staticmethod
    def _sigmoid(z):
        return 1.0 / (1.0 + np.exp(-np.clip(z, -250, 250)))
        
    def fit(self, X, y):
        n_samples, n_features = X.shape
        # Khởi tạo trọng số ngẫu nhiên chuẩn
        self.w = np.random.randn(n_features) * 0.01
        self.b = 0.0
        self.losses = []
        
        for epoch in range(self.n_epochs):
            # Lan truyền tiến (Forward pass)
            z = np.dot(X, self.w) + self.b
            y_hat = self._sigmoid(z)
            
            # Tính hàm mất mát Binary Cross-Entropy
            eps = 1e-15
            loss = -np.mean(y * np.log(y_hat + eps) + (1 - y) * np.log(1 - y_hat + eps))
            self.losses.append(loss)
            
            # Tính đạo hàm gradient
            dw = (1.0 / n_samples) * np.dot(X.T, (y_hat - y))
            db = (1.0 / n_samples) * np.sum(y_hat - y)
            
            # Cập nhật tham số (Gradient Descent)
            self.w -= self.lr * dw
            self.b -= self.lr * db
            
    def predict_proba(self, X):
        z = np.dot(X, self.w) + self.b
        return self._sigmoid(z)
        
    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)

stat_model = StatisticalLogisticClassifier(lr=0.1, n_epochs=1200)
stat_model.fit(X_train, y_train)

stat_preds_test = stat_model.predict(X_test)
stat_acc = np.mean(stat_preds_test == y_test)
print(f"[Statistical AI] Trọng số học được: w = {stat_model.w}, b = {stat_model.b:.4f}")
print(f"[Statistical AI] Độ chính xác trên tập kiểm thử: {stat_acc * 100:.2f}%")

## 4. Hệ thống Lai (Neuro-Symbolic Hybrid AI)

Hệ thống lai sử dụng sức mạnh học quy luật thống kê từ dữ liệu của mô hình học máy, nhưng áp dụng các **chốt chặn an toàn (Guardrail Rules)** bất khả xâm phạm từ logic ký hiệu:
- Nếu khách hàng có lịch sử tín dụng quá kém ($X_2 < 1.5$) hoặc thu nhập tiệm cận $0$ ($X_1 < 1.0$), hệ thống ký hiệu kích hoạt luật phủ quyết (Override): **Từ chối ngay lập tức bất chấp dự đoán của mô hình thống kê**.
- Trong vùng dữ liệu thông thường, hệ thống tin cậy vào ước lượng xác suất của mô hình thống kê.

In [ ]:
class NeuroSymbolicHybridSystem:
    """
    Hệ thống lai Neuro-Symbolic kết hợp dự đoán xác suất và luật ràng buộc an toàn.
    """
    def __init__(self, statistical_model):
        self.stat_model = statistical_model
        
    def predict(self, X):
        stat_preds = self.stat_model.predict(X)
        final_preds = []
        
        for idx, x in enumerate(X):
            x1, x2 = x[0], x[1]
            # Luật ký hiệu bảo vệ an toàn (Safety Guardrails)
            if x2 < 1.5 or x1 < 1.0:
                final_preds.append(1) # Cưỡng chế từ chối
            elif x1 >= 8.5 and x2 >= 8.5:
                final_preds.append(0) # Cưỡng chế duyệt vì khách hàng siêu uy tín
            else:
                final_preds.append(stat_preds[idx])
                
        return np.array(final_preds)

hybrid_model = NeuroSymbolicHybridSystem(stat_model)
hyb_preds_test = hybrid_model.predict(X_test)
hyb_acc = np.mean(hyb_preds_test == y_test)
print(f"[Neuro-Symbolic AI] Độ chính xác trên tập kiểm thử: {hyb_acc * 100:.2f}%")

## 5. Trực quan hóa so sánh Ranh giới quyết định (Decision Boundaries)

In [ ]:
# Tạo lưới điểm 2D để vẽ ranh giới quyết định
x1_grid, x2_grid = np.meshgrid(np.linspace(0, 10, 200), np.linspace(0, 10, 200))
grid_points = np.c_[x1_grid.ravel(), x2_grid.ravel()]

# Dự đoán trên lưới
grid_sym = symbolic_model.predict(grid_points).reshape(x1_grid.shape)
grid_stat = stat_model.predict(grid_points).reshape(x1_grid.shape)
grid_hyb = hybrid_model.predict(grid_points).reshape(x1_grid.shape)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

titles = [
    f"Symbolic AI (Rule-Based)\nAcc: {sym_acc*100:.1f}%",
    f"Statistical AI (Logistic)\nAcc: {stat_acc*100:.1f}%",
    f"Neuro-Symbolic Hybrid\nAcc: {hyb_acc*100:.1f}%"
]
grids = [grid_sym, grid_stat, grid_hyb]

for i, ax in enumerate(axes):
    # Vẽ vùng quyết định (Màu đỏ: Rủi ro, Màu xanh: An toàn)
    ax.contourf(x1_grid, x2_grid, grids[i], alpha=0.3, levels=[-0.5, 0.5, 1.5], colors=['#2ecc71', '#e74c3c'])
    # Vẽ các điểm dữ liệu kiểm thử
    ax.scatter(X_test[y_test == 0, 0], X_test[y_test == 0, 1], color='#27ae60', edgecolors='k', label='Thực tế: An toàn (0)')
    ax.scatter(X_test[y_test == 1, 0], X_test[y_test == 1, 1], color='#c0392b', edgecolors='k', label='Thực tế: Rủi ro (1)')
    ax.set_title(titles[i], fontsize=13, fontweight='bold')
    ax.set_xlabel("Thu nhập chuẩn hóa (X1)", fontsize=11)
    ax.set_ylabel("Điểm tín dụng chuẩn hóa (X2)", fontsize=11)
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 10)
    if i == 0:
        ax.legend(loc='upper right')

plt.suptitle("So sánh Ranh giới Quyết định: Ký hiệu vs Thống kê vs Lai", fontsize=15, y=1.03)
plt.tight_layout()
plt.show()

## 6. Kiểm tra Thực nghiệm: Độ giòn và Kháng nhiễu (Brittleness & Robustness Analysis)

Thử nghiệm khi dữ liệu kiểm thử bị tăng độ nhiễu lên 30% và xuất hiện các mẫu ngoại lai (Out-of-Distribution - OOD).

In [ ]:
# Sinh dữ liệu kiểm thử có nhiễu cao (30% nhãn bị xáo trộn)
X_noisy, y_noisy = generate_credit_data(n_samples=200, noise_ratio=0.30)

acc_sym_noisy = np.mean(symbolic_model.predict(X_noisy) == y_noisy)
acc_stat_noisy = np.mean(stat_model.predict(X_noisy) == y_noisy)
acc_hyb_noisy = np.mean(hybrid_model.predict(X_noisy) == y_noisy)

print("=== KẾT QUẢ TRÊN DỮ LIỆU NHIỄU CAO (30% NOISE) ===")
print(f"Symbolic AI Acc       : {acc_sym_noisy * 100:.2f}%")
print(f"Statistical AI Acc    : {acc_stat_noisy * 100:.2f}%")
print(f"Neuro-Symbolic Acc    : {acc_hyb_noisy * 100:.2f}%")

## 7. Tổng kết bài học thực hành

1. **AI Ký hiệu (Symbolic AI):**
   - Tạo ra các đường biên quyết định dạng góc cạnh, phân mảnh dựa trên các ngưỡng rời rạc do con người áp đặt.
   - Tính minh bạch 100% (luôn giải thích được tại sao một hồ sơ bị từ chối).
   - Nhược điểm: Cực kỳ "giòn gãy" (brittle), không thể xử lý mượt mà sự biến thiên liên tục trong thế giới thực.

2. **AI Thống kê (Statistical AI):**
   - Tạo ra ranh giới quyết định liên tục, mượt mà dựa trên tối ưu hóa xác suất.
   - Kháng nhiễu tốt hơn và tự động tìm ra trọng số tối ưu từ dữ liệu mà không cần con người đoán trước ngưỡng.
   - Nhược điểm: Tính giải thích mờ đục hơn hệ ký hiệu; có thể đưa ra quyết định sai sót ngớ ngẩn ở các điểm biên nếu không có ràng buộc nghiệp vụ.

3. **Hệ lai (Neuro-Symbolic AI):**
   - Đại diện cho xu hướng hiện đại: Tận dụng khả năng nhận diện hình mẫu mềm dẻo của mạng nơ-ron/thống kê, đồng thời thiết lập "lồng an toàn" bằng các luật logic cứng.